# 04 Selection (LLM filter on the B3 list)

**What this does.** For each target, an LLM reads the target's B3 comp list (names, websites, Sonnet facets and
Fiona's sub-sector; no ranks, similarity, labels or figures) and answers good (1) or bad (0) for every comp, with a
one-sentence reason. One call per target. The kept comps are the B3 + LLM filter route; B3 alone is the other.
This step chooses comps; judges (07) only evaluate.

**Current run (2026-11-18):** Claude Sonnet 5.5 through the Claude Code CLI on the Claude subscription (no API
key), prompt `2026-11-04a` (company size left out), on the current list "Sonnet + Fiona sub-sector": 373 targets,
2,538 pairs.

**Saved outputs:** `outputs-local/v1a/04_selection/`. Each call's prompt and raw answer are kept under `runs/`.

**Changelog**
- 2026-11-18: created. First selector run on the current list (until now only the 11-04 runs on the Haiku-facet
  list existed; they stay in the v1 record and are reproduced below as a check).

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v1a" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v1a" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

## 1. Current setting (`v1a/current.toml`)

In [2]:
pd.Series(cur["selection"])

route                                                                                       cosine_then_llm
model                                                                                     claude-sonnet-5-5
prompt_version                                                                                  2026-11-04a
backend                                                                                          claude-cli
candidates                                               outputs-local/v1a/03_retrieval/selector_candidates
judgments         outputs-local/v1a/04_selection/b3_selector_candidates_2026-11-04a_claude-cli_claude-so...
dtype: object

## 2. Inputs

In [3]:
run_dir = paths.path(cur["selection"]["judgments"]).parent.parent
nb.inputs([cur["retrieval"]["candidates"] + "/candidates.csv", cur["selection"]["judgments"]])

,input,rows,sha256,status
0,outputs-local/v1a/03_retrieval/selector_candidates/candidates.csv,3730,1544b96bfc27,matches seed
1,outputs-local/v1a/04_selection/b3_selector_candidates_2026-11-04a_claude-cli_claude-so...,2538,dec35ac48af2,new in v1a


## 3. Results

### 3.1 Coverage and keep rate

In [4]:
j = pd.read_csv(paths.path(cur["selection"]["judgments"]), dtype={"target": str, "comp": str})
cand = pd.read_csv(paths.path(cur["retrieval"]["candidates"]) / "candidates.csv", dtype={"target": str, "comp": str})
sel = cand[cand["selected"]]
print(f"{j['target'].nunique()} targets answered, {len(j)} pairs; model {j['model'].unique().tolist()}, prompt {j['prompt_version'].unique().tolist()}")
kept = j.groupby("target")["score"].sum()
pd.Series({"pairs kept (good)": int(j["score"].sum()), "pairs dropped (bad)": int((j["score"] == 0).sum()),
           "keep rate": round(j["score"].mean(), 3), "targets with no comp kept": int((kept == 0).sum()),
           "targets with 3+ comps kept": int((kept >= 3).sum())}).to_frame("value")

373 targets answered, 2538 pairs; model ['claude-sonnet-5-5'], prompt ['2026-11-04a']


,value
pairs kept (good),507.0
pairs dropped (bad),2031.0
keep rate,0.2
targets with no comp kept,108.0
targets with 3+ comps kept,64.0


In [5]:
pd.DataFrame({"comps kept": kept.value_counts().sort_index()}).T

score,0,1,2,3,4,5,6,7
comps kept,108,127,74,39,13,10,1,1


### 3.2 Keep rate by B3 rank and by how the comp entered the list

In [6]:
m = j.merge(sel[["target", "comp", "mutual", "topped_up"]], on=["target", "comp"], how="left")
display(m.groupby("rank")["score"].agg(["count", "mean"]).rename(columns={"count": "pairs", "mean": "keep rate"}).round(3).T)
m.assign(entry=m["topped_up"].map({True: "topped up", False: "mutual top 10"})).groupby("entry")["score"].agg(["count", "mean"]).round(3)

rank,1,2,3,4,5,6,7,8,9,10
pairs,367.000,358.000,328.000,290.000,260.000,242.000,215.000,174.000,165.000,139.000
keep rate,0.638,0.349,0.207,0.141,0.077,0.037,0.014,0.017,0.012,0.014


,count,mean
entry,,
mutual top 10,2492,0.203
topped up,46,0.022


### 3.3 Examples

In [7]:
for t in ["PCTY", "WEN", "THR"]:
    print(t); display(j[j["target_ticker"] == t][["rank", "comp_ticker", "score", "reason"]].sort_values("rank"))

PCTY


,rank,comp_ticker,score,reason
778,1,ASUR,1,"Cloud payroll, HR, time and talent software with tax filing and client-fund interest, ..."
779,2,PAYC,1,Single-platform cloud HCM and payroll subscription software sold directly to mid-sized...
780,3,ADP,1,"Cloud payroll and HCM software with tax filing and client-fund interest, selling to mi..."
781,4,ALIT,0,Outsourced benefits administration and consulting for large employers; services-heavy ...
782,5,BBSI,0,"Regional PEO and staffing business earning co-employment fees and workers' comp, not s..."
783,7,INTU,0,Mostly consumer tax and small-business accounting software; payroll is a minor line wi...
784,10,BLKB,0,"Cloud software for nonprofits and schools (fundraising, school management), not payrol..."


WEN


,rank,comp_ticker,score,reason
576,1,BROS,1,"Drive-thru quick-service chain mixing franchise royalties and company-operated units, ..."
577,2,TXRH,0,Company-operated full-service casual dining steakhouses; different dining format and c...
578,3,STKS,0,"Upscale full-service restaurants, bars and hotel management; different format, custome..."
579,4,HQI,0,Franchised temporary staffing sold to businesses; shares franchise royalties but serve...
580,5,PLNT,0,Franchised budget gyms earning membership and equipment revenue; similar franchise mod...
581,6,BDL,0,Casual full-service sports bar restaurants plus liquor retail in South Florida; differ...
582,7,LW,0,"Manufacturer of frozen potato products sold B2B; a supplier to quick-service chains, n..."


THR


,rank,comp_ticker,score,reason
18,1,FLS,0,"Pumps, valves and seals with aftermarket service; same end customers but different flo..."
19,2,PPIH,1,"Makes heat tracing and pre-insulated piping for oil and gas and industrial projects, w..."
20,3,SPXC,0,"HVAC cooling towers, boilers and locators; mostly building HVAC and detection, only pa..."
21,4,POWL,0,Custom switchgear and power control rooms; similar project model and customers but dif...
22,5,CR,0,"Aerospace sensors plus valves and pumps; no heat tracing or heating systems, so a diff..."
23,6,IESC,0,Mainly electrical contracting for data centers and homebuilders; heating products are ...
24,8,ESAB,0,Welding and cutting equipment and consumables; different products and distribution-dri...
25,9,CGEH,0,Microturbine power generation and rentals; different product and economics from indust...
26,10,MSA,0,Personal safety and gas detection equipment; shares industrial customers but not heati...


### 3.4 Against Barbara's 293 reviewed pairs

Her 0/1/2 grades (rounds 2-3, old form) on the same list. One reviewer, not ground truth; the agent-ranking review
is pending. Read from the seeded review table; it moves to the `dev_293` cohort when 06 is ported.

In [8]:
labels = pd.read_csv(paths.out("06_labels") / "barbara_293" / "review" / "pairs.csv", dtype={"target_id": str, "comp_id": str})
lab = labels.merge(j, left_on=["target_id", "comp_id"], right_on=["target", "comp"], how="left")
print(f"{lab['score'].notna().sum()} of {len(labels)} reviewed pairs have a selector answer")
ct = pd.crosstab(lab["score_barbara"], lab["score"].map({1: "kept", 0: "dropped"}), margins=True)
display(ct)
lab.groupby("score_barbara")["score"].mean().round(3).rename("keep rate").to_frame().T

293 of 293 reviewed pairs have a selector answer


score,dropped,kept,All
score_barbara,,,
0,88,3,91
1,136,42,178
2,0,24,24
All,224,69,293


score_barbara,0,1,2
keep rate,0.033,0.236,1.0


## 4. Checks

In [9]:
import cosine_then_llm as ctl, check_prompt_versions as cpv
checks.add("every target answered", j["target"].nunique() == 373, f"{j['target'].nunique()} targets")
a = set(zip(sel["target"], sel["comp"])); b = set(zip(j["target"], j["comp"]))
checks.add("every selected B3 pair answered exactly once", a == b and not j.duplicated(["target", "comp"]).any(), f"{len(j)} pairs")
checks.add("no answer issues", j["issues"].fillna("").eq("").all(), str(j["issues"].fillna("").ne("").sum()))
checks.add("prompt 2026-11-04a, Sonnet", j["prompt_version"].eq(ctl.PROMPT_VERSION).all() and j["model"].eq(cur["selection"]["model"]).all())
import contextlib, io
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    jt = cpv.check_judge(); dt = cpv.check_direct()
checks.add("v1 record: saved selector prompts rebuild (373/373)", jt[0] == jt[1] == 373, str(jt))
checks.add("v1 record: saved direct-pick prompts rebuild (373/373)", dt[0] == dt[1] == 373, str(dt))
checks.show()

6/6 checks pass


,check,result,detail
0,every target answered,pass,373 targets
1,every selected B3 pair answered exactly once,pass,2538 pairs
2,no answer issues,pass,0
3,"prompt 2026-11-04a, Sonnet",pass,
4,v1 record: saved selector prompts rebuild (373/373),pass,"(373, 373, 0)"
5,v1 record: saved direct-pick prompts rebuild (373/373),pass,"(373, 373, 0)"


## 5. History (v1)

- **v0 (10-25/27):** Haiku 5.5 screened v0's cosine z >= 2 lists (prompt `2026-10-25a`; `outputs-local/v1/judge/`),
  and Haiku picked comps directly from the pool (`outputs-local/v1/suggest/`). GPT-6.1 Sol graded pairs as the judge.
- **11-04 (Matthew, Richard):** Sonnet and Haiku selectors on the B3 list built from Haiku facets, reading Haiku
  facets (`model_selection_eval.py`, reproduced exactly in v1a). Both strict: about 21% (Sonnet) and 17% (Haiku) of
  pairs kept; the direct picks were where the model mattered.
- **11-05 (Eric):** the current list was built and its 373 prompts written as a dry run; no answers until 2026-11-18.
- **Kethan's pipeline (11-12, on his machine):** GPT-6.1 Sol as selector on his own Sol-filled B3 list.